In [7]:
"""
Final Evaluation Script for the Action-Masked RL Agent (v6.3 - Relocated)

v6.3: Paths have been updated to run correctly from the `notebooks/` directory.
"""
# --- Imports and Path Setup ---
import torch, numpy as np, pandas as pd, gymnasium as gym, os, sys
from collections import Counter
from sb3_contrib import MaskablePPO

# This path logic correctly adds the project's root directory to the system path,
# allowing us to import from the 'src' folder.
try:
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
except NameError:
    sys.path.append(os.path.abspath('..'))

# CORRECTED: The import path now explicitly looks inside the 'src.training' module
from src.training.train_rl_agent_sepsis import SepsisEnv, Config, TransformerPolicy

# --- Evaluation Function ---
def evaluate_sepsis_agent():
    print("--- Evaluating Final Masked Sepsis Agent ---")
    config = Config()
    PANEL_NAMES = {0: "CBC", 1: "CMP", 2: "ABG", 3: "aPTT"}
    
    # CORRECTED: Added '..' to the path to go up one level from `notebooks` to the root
    agent_path = os.path.join("..", config.MODEL_DIR, "rl_agent_sepsis_masked_final.zip")
    if not os.path.exists(agent_path):
        print(f"Error: Model not found at {agent_path}. Please train the agent first."); return

    # 1. Set up the Environment
    env = SepsisEnv(config)
    
    # CORRECTED: Added '..' to the paths for the data files
    data_path_prefix = os.path.join("..", config.PROCESSED_DATA_DIR)
    env.X_val = pd.read_csv(os.path.join(data_path_prefix, "test_X.csv"))
    env.y_val = pd.read_csv(os.path.join(data_path_prefix, "test_y.csv"))
    
    env.num_patients = len(env.X_val)
    print(f"Loaded TEST data with {env.num_patients} patients.")
    
    # 2. Load the Trained Agent
    model = MaskablePPO.load(agent_path)
    print("Trained agent loaded successfully.")

    # 3. Initialize Metrics
    total_rewards, total_costs, num_correct_diagnoses = [], [], 0
    test_usage_counter = Counter()

    # 4. Run Evaluation Loop
    for i in range(env.num_patients):
        obs, info = env.reset()
        done = False
        while not done:
            action_masks = env.action_masks()
            action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
            action = action.item()
            
            if action != env.DIAGNOSE_ACTION:
                total_costs.append(config.COST_MAPPING.get(action, 0))
                test_usage_counter[action] += 1
            
            obs, reward, done, truncated, info = env.step(action)
        
        if reward > 0: num_correct_diagnoses += 1
        total_rewards.append(reward)

    # 5. Calculate and Report Final Metrics
    avg_reward = np.mean(total_rewards)
    avg_test_panels = sum(test_usage_counter.values()) / env.num_patients
    avg_dollar_cost = sum(total_costs) / env.num_patients
    accuracy = (num_correct_diagnoses / env.num_patients) * 100

    print("\n" + "="*50 + "\n--- Final Performance Report ---\n" + "="*50)
    print(f"📈 Final Diagnostic Accuracy: {accuracy:.2f}%")
    print(f"💰 Average Financial Cost per Patient: ${avg_dollar_cost:.2f}")
    print(f"📉 Average Number of Tests Ordered: {avg_test_panels:.2f}")
    print(f"📊 Average Reward: {avg_reward:,.2f}")
    print("\n" + "="*50 + "\n--- Agent's Testing Strategy ---\n" + "="*50)
    for action_index, count in test_usage_counter.most_common():
        print(f"  - {PANEL_NAMES.get(action_index, 'Unknown')}: {count} times")
    print("="*50)

if __name__ == "__main__":
    evaluate_sepsis_agent()

--- Evaluating Final Masked Sepsis Agent ---


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/sepsis\\val_X.csv'